# Inkling 2 Mini UD-Q3 GGUF — llama.cpp chat, coding tests, and vision on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/inkling-2-mini-ud-q3-gguf-cea2/colab/Inkling-2-Mini-UD-Q3-GGUF-llamacpp.ipynb)

[`unsloth/Inkling-2-Mini-GGUF`](https://huggingface.co/unsloth/Inkling-2-Mini-GGUF) is Unsloth's UD imatrix GGUF of [Thinking Machines Lab **Inkling 2 Mini**](https://huggingface.co/thinkingmachines/Inkling-2-Mini): a **multimodal** (image + text → text) **looped MoE** released **20 November 2026** under **Apache-2.0**. The card lists **26.2B total / 3.4B active** parameters, **128 experts, top-8**, and **24** decoder layers whose **middle 12 run three times** (about 5.4B effective active parameters per token).

This notebook is **inference only**. It does **not** compile llama.cpp. Compiling CUDA llama.cpp on a free Colab T4 repeatedly disconnects the runtime (that is why [PR #82](https://github.com/donvito/notebooks/pull/82) is blocked). It downloads Unsloth's **prebuilt Linux CUDA 12** `llama-server` from [`unslothai/llama.cpp` releases](https://github.com/unslothai/llama.cpp/releases), then loads the **root** UD GGUF (Unsloth looped layout). Do **not** use `Mainline_Support/*.gguf_file` here — those copies need mainline llama.cpp plus unmerged [PR #32704](https://github.com/ggml-org/llama.cpp/pull/32704), which would require a source build.

**Primary quant:** `Inkling-2-Mini-UD-Q3_K_XL.gguf` (**13.63 GB**, mean KLD **0.0174** vs UD-Q8_K_XL). **Vision projector:** `mmproj-F16.gguf` (**0.64 GB**). A free T4 has ~15 GB VRAM (~14.5 GB usable), so full GPU offload plus the projector plus an 8K KV cache will **not** fit. The server uses `-ngl 99` with **`--n-cpu-moe 16`** so the last 16 MoE layers keep their expert weights in system RAM (Colab has ~12.7 GB RAM; mmap, **no `--mlock`**). If that path OOMs, switch `QUANT` to **`UD-IQ3_XXS`** (11.66 GB) and rerun from the download cell.

**Sampling** (Unsloth / card): temperature **1.0**, top-p **1.0**, top-k **0**, min-p **0**.

See [Use cases](https://github.com/donvito/notebooks/blob/melvin/inkling-2-mini-ud-q3-gguf-cea2/docs/Inkling-2-Mini-UD-Q3-GGUF-llamacpp.md) for when this notebook is a good fit.

**Licence:** Apache-2.0. The Hub repo is public — cancel any Colab `HF_TOKEN` prompt; a token is optional for higher rate limits.

**Runtime:** *Runtime > Change runtime type > T4 GPU*.

**Links:** [Unsloth GGUF](https://huggingface.co/unsloth/Inkling-2-Mini-GGUF) · [base card](https://huggingface.co/thinkingmachines/Inkling-2-Mini) · [Unsloth run guide](https://unsloth.ai/docs/models/inkling-2-mini) · [prebuilt llama.cpp](https://github.com/unslothai/llama.cpp/releases) · [use cases](https://github.com/donvito/notebooks/blob/melvin/inkling-2-mini-ud-q3-gguf-cea2/docs/Inkling-2-Mini-UD-Q3-GGUF-llamacpp.md)


## 1. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. A free T4 (~15 GB, Turing `sm_75`) is the path this notebook is written for. The next cells stop if you are on CPU.


In [ ]:
import os
import shutil
import subprocess

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Public repo: skip Colab userdata.get("HF_TOKEN") popups. A token is optional.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
print("GPU name:", name, f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print()
print("=== T4-first notes ===")
print("- Do NOT compile llama.cpp. Use Unsloth's CUDA 12 prebuilt (next cell).")
print("- UD-Q3_K_XL is 13.63 GB + mmproj-F16 0.64 GB. Full -ngl 99 will not fit.")
print("- Default --n-cpu-moe 16 keeps the last 16 MoE layers' experts in RAM (mmap, no --mlock).")
print("- If Q3_K_XL OOMs, set QUANT = \"UD-IQ3_XXS\" (11.66 GB) in the download cell.")
print("- Flash attention: -fa auto (llama.cpp decides; Turing has no PyTorch FA2).")
if shutil.which("nvidia-smi"):
    subprocess.check_call(["nvidia-smi"])


## 2. Download Unsloth's llama.cpp CUDA 12 prebuilt

Resolve the latest [`unslothai/llama.cpp`](https://github.com/unslothai/llama.cpp/releases) tag via the GitHub API, with pinned fallback **`b12588-mix-c0c3de8`**. Read `llama-prebuilt-manifest.json` and pick the Linux x64 **CUDA 12** asset whose `supported_sms` includes **75** (Turing / T4): prefer `*-cuda12-portable.tar.gz`, else `*-cuda12-older.tar.gz`.

This cell never runs `cmake` or `make`.


In [ ]:
%pip install -qqq --no-cache-dir nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 "huggingface_hub>=0.34"

import glob
import json
import os
import shutil
import tarfile
import urllib.error
import urllib.request

FALLBACK_TAG = "b12588-mix-c0c3de8"
RELEASES_API = "https://api.github.com/repos/unslothai/llama.cpp/releases/latest"
ASSET_PREFIX = "https://github.com/unslothai/llama.cpp/releases/download"
REQUIRED_SM = "75"
ROOT = "/content" if os.path.isdir("/content") else "/tmp"
BIN_DIR = os.path.join(ROOT, "llama.cpp")
ARCHIVE = os.path.join(ROOT, "llama-unsloth-cuda12.tgz")


def _http_json(url: str) -> dict:
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.github+json", "User-Agent": "donvito-notebooks"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode())


def _download(url: str, dest: str) -> None:
    print("downloading", url)
    urllib.request.urlretrieve(url, dest)


def resolve_cuda12_sm75_asset(tag: str) -> tuple[str, str, list[str]]:
    """Return (asset_name, url, supported_sms) for linux-x64 CUDA 12 covering sm_75."""
    manifest_url = f"{ASSET_PREFIX}/{tag}/llama-prebuilt-manifest.json"
    manifest = _http_json(manifest_url)
    candidates = []
    for art in manifest.get("artifacts") or []:
        if art.get("install_kind") != "linux-cuda" or art.get("runtime_line") != "cuda12":
            continue
        sms = [str(s) for s in (art.get("supported_sms") or [])]
        if REQUIRED_SM not in sms:
            continue
        name = art.get("asset_name") or ""
        if not name.endswith(".tar.gz"):
            continue
        rank = 0 if art.get("coverage_class") == "portable" else 1
        candidates.append((rank, name, sms, art.get("coverage_class")))
    if not candidates:
        raise SystemExit(
            f"No linux-x64 CUDA 12 asset with sm_{REQUIRED_SM} in {manifest_url}. "
            "Do not compile llama.cpp; pick another Unsloth release tag."
        )
    candidates.sort()
    _, name, sms, kind = candidates[0]
    url = f"{ASSET_PREFIX}/{tag}/{name}"
    print(f"manifest {tag}: using {name} ({kind}, sms={sms})")
    return name, url, sms


try:
    latest = _http_json(RELEASES_API)
    LLAMA_TAG = latest.get("tag_name") or FALLBACK_TAG
    print("GitHub latest unslothai/llama.cpp tag:", LLAMA_TAG)
except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
    LLAMA_TAG = FALLBACK_TAG
    print(f"GitHub API failed ({exc}); falling back to {FALLBACK_TAG}")

try:
    ASSET_NAME, LLAMA_URL, ASSET_SMS = resolve_cuda12_sm75_asset(LLAMA_TAG)
except Exception as exc:
    print(f"Could not use {LLAMA_TAG} ({exc}); falling back to {FALLBACK_TAG}")
    LLAMA_TAG = FALLBACK_TAG
    ASSET_NAME, LLAMA_URL, ASSET_SMS = resolve_cuda12_sm75_asset(LLAMA_TAG)

server_guess = os.path.join(BIN_DIR, "llama-server")
if not os.path.exists(server_guess):
    _download(LLAMA_URL, ARCHIVE)
    shutil.rmtree(BIN_DIR, ignore_errors=True)
    os.makedirs(BIN_DIR, exist_ok=True)
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall(BIN_DIR)
    nested = [os.path.join(BIN_DIR, name) for name in os.listdir(BIN_DIR)]
    if len(nested) == 1 and os.path.isdir(nested[0]) and not os.path.exists(os.path.join(BIN_DIR, "llama-server")):
        for name in os.listdir(nested[0]):
            os.rename(os.path.join(nested[0], name), os.path.join(BIN_DIR, name))


def find_llama_server(root: str) -> str:
    for dirpath, _, files in os.walk(root):
        if "llama-server" in files:
            path = os.path.join(dirpath, "llama-server")
            os.chmod(path, os.stat(path).st_mode | 0o111)
            return path
    raise SystemExit(f"llama-server missing after extract under {root}")


LLAMA_SERVER = find_llama_server(BIN_DIR)
os.chmod(LLAMA_SERVER, os.stat(LLAMA_SERVER).st_mode | 0o111)
print("tag", LLAMA_TAG)
print("asset", ASSET_NAME, "sms", ASSET_SMS)
print("llama-server", LLAMA_SERVER)


## 3. Download the GGUF (Q3_K_XL + mmproj only)

`hf download` pulls **only** the two files this notebook needs from [`unsloth/Inkling-2-Mini-GGUF`](https://huggingface.co/unsloth/Inkling-2-Mini-GGUF). Colab disk is enough (~14.3 GB). Do **not** download `Mainline_Support/`.

**Knobs** (edit here, then rerun this cell and the server cell):

- `QUANT`: default **`UD-Q3_K_XL`**. If `llama-server` OOMs on the T4, set **`UD-IQ3_XXS`**.
- `N_CPU_MOE`: default **16**. Inkling 2 Mini has **24** unique decoder layers, all MoE. Raise toward **24** if VRAM OOMs; lower toward **8** if `nvidia-smi` shows headroom and you want more experts on the GPU (faster).


In [ ]:
import os
import shutil
import subprocess

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# --- knobs ---
REPO_ID = "unsloth/Inkling-2-Mini-GGUF"
QUANT = "UD-Q3_K_XL"  # 13.63 GB. Fallback: "UD-IQ3_XXS" (11.66 GB) if this OOMs on T4.
MMPROJ_FILE = "mmproj-F16.gguf"  # 0.64 GB
N_CPU_MOE = 16  # last N MoE layers keep expert weights on CPU (mmap, no --mlock)
N_CTX = 8192
N_GPU_LAYERS = 99
HOST = "127.0.0.1"
PORT = 8088
SERVER_URL = f"http://{HOST}:{PORT}"
SAMPLER = dict(temperature=1.0, top_p=1.0, top_k=0, min_p=0.0)
# --- end knobs ---

QUANT_FILES = {
    "UD-Q3_K_XL": ("Inkling-2-Mini-UD-Q3_K_XL.gguf", 13.63),
    "UD-IQ3_XXS": ("Inkling-2-Mini-UD-IQ3_XXS.gguf", 11.66),
}
if QUANT not in QUANT_FILES:
    raise SystemExit(f"QUANT must be one of {sorted(QUANT_FILES)}")
MODEL_FILE, QUANT_GB = QUANT_FILES[QUANT]
if QUANT != "UD-Q3_K_XL":
    print(
        f"NOTE: using fallback quant {QUANT} ({QUANT_GB} GB) instead of UD-Q3_K_XL "
        "(13.63 GB). Use this when the 3-bit XL path cannot fit on the T4."
    )

LOCAL_DIR = os.path.join(ROOT, "Inkling-2-Mini-GGUF")
os.makedirs(LOCAL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(LOCAL_DIR, MODEL_FILE)
MMPROJ_PATH = os.path.join(LOCAL_DIR, MMPROJ_FILE)

hf_bin = shutil.which("hf") or shutil.which("huggingface-cli")
if not hf_bin:
    raise SystemExit("hf / huggingface-cli not on PATH. Re-run the install cell.")

dl_cmd = [hf_bin, "download", REPO_ID, MODEL_FILE, MMPROJ_FILE, "--local-dir", LOCAL_DIR]
print("running", " ".join(dl_cmd))
subprocess.run(dl_cmd, check=True, env={**os.environ, "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"})

if not os.path.exists(MODEL_PATH) or not os.path.exists(MMPROJ_PATH):
    raise SystemExit(f"Expected {MODEL_PATH} and {MMPROJ_PATH} after hf download")

print("model", MODEL_PATH, f"({os.path.getsize(MODEL_PATH) / 1024**3:.2f} GB)")
print("mmproj", MMPROJ_PATH, f"({os.path.getsize(MMPROJ_PATH) / 1024**3:.2f} GB)")
print("QUANT", QUANT, "| N_CPU_MOE", N_CPU_MOE, "| N_CTX", N_CTX, "| -ngl", N_GPU_LAYERS)
print("sampler", SAMPLER)


## 4. Start `llama-server` and wait for `/health`

`--mmproj` enables the image demo. `--jinja` uses the GGUF chat template. `-fa auto` lets this Unsloth build use flash attention when the kernels exist. `--n-cpu-moe` is the T4 fit knob from the previous cell. mmap stays on; `--mlock` is not passed.

First load of a 13.6 GB MoE can take several minutes. This cell waits up to 12 minutes for `/health`.


In [ ]:
import glob
import importlib
import os
import subprocess
import time

import requests

LIBS = []
for pattern in (
    "/usr/local/cuda*/lib64",
    "/usr/local/cuda*/targets/x86_64-linux/lib",
    "/usr/lib/x86_64-linux-gnu",
    "/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib",
    "/usr/local/lib/python*/dist-packages/nvidia/cublas/lib",
    "/usr/local/lib/python*/dist-packages/nvidia/nvjitlink/lib",
):
    LIBS.extend(path for path in glob.glob(pattern) if os.path.isdir(path))
for pkg_name in ("nvidia.cuda_runtime", "nvidia.nvjitlink", "nvidia.cublas"):
    try:
        pkg = importlib.import_module(pkg_name)
    except ImportError:
        continue
    lib_dir = os.path.join(pkg.__path__[0], "lib")
    if os.path.isdir(lib_dir) and lib_dir not in LIBS:
        LIBS.append(lib_dir)

ENV = os.environ.copy()
ENV["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
ENV["LD_LIBRARY_PATH"] = ":".join(
    [os.path.dirname(LLAMA_SERVER), BIN_DIR, *LIBS, ENV.get("LD_LIBRARY_PATH", "")]
)


def print_vram(label=""):
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")
    used = smi.split(",")[0].strip()
    PEAK_VRAM.append((label or "smi", used))


PEAK_VRAM = []
RUNS = []
print_vram("before server")

LOG = os.path.join(ROOT, "llama-server.log")
SERVER_CMD = [
    LLAMA_SERVER,
    "--model", MODEL_PATH,
    "--mmproj", MMPROJ_PATH,
    "--jinja",
    "-ngl", str(N_GPU_LAYERS),
    "--n-cpu-moe", str(N_CPU_MOE),
    "-c", str(N_CTX),
    "-fa", "auto",
    "--parallel", "1",
    "--temp", str(SAMPLER["temperature"]),
    "--top-p", str(SAMPLER["top_p"]),
    "--top-k", str(SAMPLER["top_k"]),
    "--min-p", str(SAMPLER["min_p"]),
    "--host", HOST,
    "--port", str(PORT),
]


def stop_llama_server(timeout=15):
    global server_proc, log_f
    if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait(timeout=5)
        print("llama-server stopped")
    else:
        print("llama-server was not running")
    if "log_f" in globals() and log_f is not None and not getattr(log_f, "closed", True):
        log_f.close()


def wait_for_health(timeout=720):
    deadline = time.time() + timeout
    last_err = None
    while time.time() < deadline:
        if server_proc.poll() is not None:
            if "log_f" in globals() and log_f is not None and not getattr(log_f, "closed", True):
                log_f.flush()
            tail = open(LOG).read()[-4000:]
            hint = ""
            if QUANT == "UD-Q3_K_XL" and any(
                token in tail.lower() for token in ("out of memory", "oom", "failed to allocate")
            ):
                hint = (
                    "\n\nQ3_K_XL did not fit. Set QUANT = \"UD-IQ3_XXS\" in the download cell "
                    "(and/or raise N_CPU_MOE toward 24) and rerun from there."
                )
            raise SystemExit(
                f"llama-server exited {server_proc.returncode}. Tail of {LOG}:\n{tail}{hint}"
            )
        try:
            health = requests.get(f"{SERVER_URL}/health", timeout=3)
            if health.ok:
                print("llama-server health:", health.text.strip() or health.status_code)
                return
            last_err = f"HTTP {health.status_code} {health.text[:200]}"
        except requests.RequestException as exc:
            last_err = str(exc)
        time.sleep(2)
    raise SystemExit(f"llama-server did not become healthy: {last_err}\n" + open(LOG).read()[-4000:])


def start_llama_server():
    global server_proc, log_f
    stop_llama_server(timeout=10)
    log_f = open(LOG, "w")
    print("starting", " ".join(SERVER_CMD))
    print(
        "Expert offload: --n-cpu-moe",
        N_CPU_MOE,
        "(last N MoE layers on CPU). Raise if OOM; lower if VRAM has headroom.",
    )
    server_proc = subprocess.Popen(
        SERVER_CMD,
        env=ENV,
        stdout=log_f,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
    )
    wait_for_health()
    print("server", SERVER_URL, "| quant", QUANT, "| n-cpu-moe", N_CPU_MOE, "| ctx", N_CTX)
    print_vram("after load")


start_llama_server()


## 5. OpenAI-compatible helper

Demos talk to `POST /v1/chat/completions`. Completions record tokens, wall time, and tokens/sec (llama.cpp `timings.predicted_per_second` when present, else completion tokens / wall seconds).


In [ ]:
import time

import requests


def chat(messages, max_tokens: int = 512, timeout: int = 600) -> dict:
    payload = {
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": SAMPLER["temperature"],
        "top_p": SAMPLER["top_p"],
        "top_k": SAMPLER["top_k"],
        "min_p": SAMPLER["min_p"],
    }
    t0 = time.perf_counter()
    response = requests.post(f"{SERVER_URL}/v1/chat/completions", json=payload, timeout=timeout)
    elapsed = time.perf_counter() - t0
    response.raise_for_status()
    data = response.json()
    choice = data["choices"][0]
    message = choice.get("message") or {}
    text = message.get("content") or ""
    reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
    usage = data.get("usage") or {}
    timings = data.get("timings") or {}
    completion_tokens = usage.get("completion_tokens") or timings.get("predicted_n") or 0
    tok_s = timings.get("predicted_per_second")
    if not tok_s and completion_tokens and elapsed:
        tok_s = completion_tokens / elapsed
    record = {
        "text": text,
        "reasoning": reasoning,
        "finish": choice.get("finish_reason"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": completion_tokens,
        "seconds": elapsed,
        "tok_s": float(tok_s) if tok_s else 0.0,
        "raw": data,
    }
    print(
        f"prompt {record['prompt_tokens']} | completion {completion_tokens} | "
        f"{elapsed:.1f} s | {record['tok_s']:.2f} tok/s | finish {record['finish']}"
    )
    print_vram("after chat")
    return record


print("chat helper ready |", SERVER_URL)


## 6. Demo 1 — text chat / reasoning

A short logic question. Completions are capped at **512** tokens so the T4 expert-offload path stays interactive.


In [ ]:
CHAT_PROMPT = (
    "There are three boxes. One contains apples, one contains oranges, and one contains "
    "apples and oranges. Every label is wrong. You pick one fruit from the box labeled "
    "'apples and oranges' and it is an apple. What is in each box? Answer in at most "
    "eight sentences."
)
print("=== prompt ===")
print(CHAT_PROMPT)
print()
print("=== reply ===")
chat_run = chat([{"role": "user", "content": CHAT_PROMPT}], max_tokens=512)
print(chat_run["text"].strip())
RUNS.append({"name": "text-chat", "passed": None, "total": None, **{k: chat_run[k] for k in ("completion_tokens", "seconds", "tok_s")}})


## 7. Coding tests

Three small tasks. Each completion is capped at **~1024** tokens. The notebook **extracts** a markdown code block (strips `<think>` / reasoning text if the model emits it), then **executes** the code against asserts and prints pass/fail per test.

- **(a)** Write `merge_intervals` from a spec, then run five asserts.
- **(b)** Find and fix a bug in a short snippet, then run five asserts on the fixed function.
- **(c)** Write a SQL `SELECT` for a tiny schema; run it with `sqlite3` in memory and check the rows.


In [ ]:
import re
import sqlite3
import traceback

THINK_RE = re.compile(
    r"<think>.*?</think>"
    r"|<\s*\|?\s*think\s*\|?\s*>.*?<\s*\|?\s*/\s*think\s*\|?\s*>"
    r"|<reasoning>.*?</reasoning>",
    re.DOTALL | re.IGNORECASE,
)
FENCE_RE = re.compile(r"```(?:([\w+-]+)\s*)?\n(.*?)```", re.DOTALL)


def strip_thinking(text: str) -> str:
    cleaned = THINK_RE.sub("", text or "")
    # Drop leftover reasoning-only prefixes that some templates leave behind.
    cleaned = re.sub(r"(?is)^\s*thinking:.*?(?=\n\S)", "", cleaned)
    return cleaned.strip()


def extract_code(text: str, prefer_langs: tuple[str, ...]) -> str:
    cleaned = strip_thinking(text)
    blocks = FENCE_RE.findall(cleaned)
    if not blocks:
        return cleaned
    preferred, others = [], []
    want = tuple(lang.lower() for lang in prefer_langs)
    for lang, body in blocks:
        body = body.strip()
        if not body:
            continue
        lang_l = (lang or "").lower()
        if any(lang_l == w or lang_l.startswith(w) for w in want):
            preferred.append(body)
        else:
            others.append(body)
    # Last matching fence is usually the final answer after drafts / thinking.
    if preferred:
        return preferred[-1]
    return others[-1] if others else cleaned


def extract_sql(text: str) -> str:
    code = extract_code(text, ("sql",))
    code = re.sub(r"(?im)^\s*--.*$", "", code)
    code = code.strip().rstrip(";")
    # If the model wrapped extra prose, keep from the first SELECT.
    match = re.search(r"(?is)\b(select\b.+)$", code)
    return (match.group(1) if match else code).strip().rstrip(";")


def run_python_tests(source: str, fn_name: str, cases: list[tuple]) -> tuple[int, int]:
    ns: dict = {}
    try:
        exec(compile(source, f"<{fn_name}>", "exec"), ns, ns)
    except Exception as exc:
        print(f"exec failed: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        return 0, len(cases)
    fn = ns.get(fn_name)
    if not callable(fn):
        print(f"extracted code has no callable {fn_name!r}. keys={list(ns)}")
        return 0, len(cases)
    passed = 0
    for i, (args, expected) in enumerate(cases, start=1):
        try:
            got = fn(*args) if isinstance(args, tuple) else fn(args)
            ok = got == expected
        except Exception as exc:
            ok = False
            got = f"{type(exc).__name__}: {exc}"
        status = "PASS" if ok else "FAIL"
        print(f"  test {i}: {status}  input={args!r}  expected={expected!r}  got={got!r}")
        passed += int(ok)
    return passed, len(cases)


print("code helpers ready")


### 7a. Write `merge_intervals` from a spec

The model must return a Python function. The notebook executes it against five asserts (LeetCode-style merge overlapping intervals).


In [ ]:
SPEC_A = """Write a Python function merge_intervals(intervals).

Spec:
- intervals is a list of [start, end] pairs of integers (start <= end).
- Merge every overlapping or touching pair (treat [1,4] and [4,5] as one interval).
- Return a new list of merged [start, end] pairs sorted by start.
- [] returns [].
- Do not use numpy. Do not read stdin. No tests in the reply.

Reply with a single ```python code block that defines merge_intervals only.
"""

print("=== prompt (a) ===")
print(SPEC_A)
print()
print("=== reply (a) ===")
run_a = chat([{"role": "user", "content": SPEC_A}], max_tokens=1024)
print(run_a["text"])
source_a = extract_code(run_a["text"] + "\n" + (run_a["reasoning"] or ""), ("python", "py"))
print()
print("=== extracted (a) ===")
print(source_a)
print()
print("=== asserts (a) ===")
cases_a = [
    ([[1, 3], [2, 6], [8, 10], [15, 18]], [[1, 6], [8, 10], [15, 18]]),
    ([[1, 4], [4, 5]], [[1, 5]]),
    ([[1, 4], [0, 4]], [[0, 4]]),
    ([[1, 4], [2, 3]], [[1, 4]]),
    ([], []),
]
passed_a, total_a = run_python_tests(source_a, "merge_intervals", cases_a)
print(f"merge_intervals: {passed_a}/{total_a} passed")
RUNS.append(
    {
        "name": "code-merge-intervals",
        "passed": passed_a,
        "total": total_a,
        "completion_tokens": run_a["completion_tokens"],
        "seconds": run_a["seconds"],
        "tok_s": run_a["tok_s"],
    }
)


### 7b. Find and fix a bug

`longest_run` below is wrong: it overwrites `best` on every reset and never takes `max` after the last run. The model should return a fixed `longest_run`. The notebook executes the fix against five asserts.


In [ ]:
BUGGY_B = """def longest_run(s):
    # Length of the longest run of the same character in s. Empty -> 0.
    if not s:
        return 0
    best = 1
    cur = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            best = cur
            cur = 1
    return best
"""

SPEC_B = f"""This Python function is buggy. Find the bug, then return a corrected longest_run.

```python
{BUGGY_B}```

Spec: longest_run(s) is the length of the longest consecutive run of the same character in s. The empty string returns 0.

Reply with a single ```python code block that defines the fixed longest_run only. No tests.
"""

print("=== prompt (b) ===")
print(SPEC_B)
print()
print("=== reply (b) ===")
run_b = chat([{"role": "user", "content": SPEC_B}], max_tokens=1024)
print(run_b["text"])
source_b = extract_code(run_b["text"] + "\n" + (run_b["reasoning"] or ""), ("python", "py"))
print()
print("=== extracted (b) ===")
print(source_b)
print()
print("=== asserts (b) ===")
cases_b = [
    (("aaabb",), 3),
    (("bbbb",), 4),
    (("baaaa",), 4),
    (("abab",), 1),
    (("",), 0),
]
passed_b, total_b = run_python_tests(source_b, "longest_run", cases_b)
print(f"longest_run (fixed): {passed_b}/{total_b} passed")
RUNS.append(
    {
        "name": "code-bugfix-longest-run",
        "passed": passed_b,
        "total": total_b,
        "completion_tokens": run_b["completion_tokens"],
        "seconds": run_b["seconds"],
        "tok_s": run_b["tok_s"],
    }
)


### 7c. SQL on a tiny in-memory table

The model writes one `SELECT`. The notebook creates `employees` in `sqlite3` (memory), runs the query, and checks department names plus rounded average salaries.


In [ ]:
SPEC_C = """Write one SQLite SELECT for this schema:

CREATE TABLE employees (
  id INTEGER PRIMARY KEY,
  name TEXT,
  dept TEXT,
  salary INTEGER
);

Return one row per department whose average salary is strictly greater than 80000.
Columns: dept, avg_salary (the average of salary).
Order by avg_salary descending.
Do not filter individual employees; average first, then filter departments.
Reply with a single ```sql code block. No INSERT statements. No comments.
"""

print("=== prompt (c) ===")
print(SPEC_C)
print()
print("=== reply (c) ===")
run_c = chat([{"role": "user", "content": SPEC_C}], max_tokens=1024)
print(run_c["text"])
sql_c = extract_sql(run_c["text"] + "\n" + (run_c["reasoning"] or ""))
print()
print("=== extracted SQL ===")
print(sql_c)

conn = sqlite3.connect(":memory:")
conn.execute(
    "CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary INTEGER)"
)
conn.executemany(
    "INSERT INTO employees (id, name, dept, salary) VALUES (?, ?, ?, ?)",
    [
        (1, "Ana", "Eng", 120000),
        (2, "Bo", "Eng", 90000),
        (3, "Cy", "HR", 70000),
        (4, "Di", "Sales", 85000),
        (5, "Ed", "Sales", 95000),
        (6, "Fa", "HR", 60000),
    ],
)
expected_c = [("Eng", 105000.0), ("Sales", 90000.0)]
passed_c = 0
total_c = 5
try:
    rows = conn.execute(sql_c).fetchall()
    print("query rows:", rows)
    got = [(str(r[0]), round(float(r[1]), 2)) for r in rows]
    exp = [(d, round(v, 2)) for d, v in expected_c]
    checks = [
        ("row count", len(got) == 2, f"got {len(got)} rows {got}"),
        ("Eng avg", ("Eng", 105000.0) in got, got),
        ("Sales avg", ("Sales", 90000.0) in got, got),
        ("HR excluded", all(d != "HR" for d, _ in got), got),
        ("order", got == exp, f"got {got} expected {exp}"),
    ]
    for name, ok, detail in checks:
        print(f"  {name}: {'PASS' if ok else 'FAIL'}  {detail}")
        passed_c += int(ok)
except Exception as exc:
    print(f"SQL exec failed: {type(exc).__name__}: {exc}")
    traceback.print_exc()
    for name in ("row count", "Eng avg", "Sales avg", "HR excluded", "order"):
        print(f"  {name}: FAIL")
conn.close()
print(f"sql-dept-avg: {passed_c}/{total_c} passed")
RUNS.append(
    {
        "name": "sql-dept-avg",
        "passed": passed_c,
        "total": total_c,
        "completion_tokens": run_c["completion_tokens"],
        "seconds": run_c["seconds"],
        "tok_s": run_c["tok_s"],
    }
)


### Coding test summary

test name, passed / total, tokens, seconds, tokens/sec.


In [ ]:
print(f"{'test':<28} {'passed':>8} {'tokens':>8} {'sec':>8} {'tok/s':>8}")
print("-" * 64)
for row in RUNS:
    if row["name"] == "text-chat":
        continue
    passed = "—" if row["passed"] is None else f"{row['passed']}/{row['total']}"
    print(
        f"{row['name']:<28} {passed:>8} {row['completion_tokens']:>8} "
        f"{row['seconds']:>8.1f} {row['tok_s']:>8.2f}"
    )
code_rows = [row for row in RUNS if row["passed"] is not None]
if code_rows:
    p = sum(row["passed"] for row in code_rows)
    t = sum(row["total"] for row in code_rows)
    print("-" * 64)
    print(f"{'TOTAL coding asserts':<28} {p}/{t}")


## 8. Demo 2 — image question

The OpenAI-compatible endpoint gets a local PNG as a `data:` URL (`--mmproj` is already on the server). Primary image: the dice PNG mirrored in this repo. Fallbacks: Wikimedia, then Hugging Face's cat-mug sample.


In [ ]:
import base64
from IPython.display import Image, display

IMAGE_URLS = [
    "https://raw.githubusercontent.com/donvito/notebooks/main/colab/data/ternary-bonsai/png-transparency-demonstration-1.png",
    "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-mug.png",
]
photo = os.path.join(ROOT, "inkling-demo.png")
headers = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
fetched = None
for url in IMAGE_URLS:
    try:
        r = requests.get(url, headers=headers, timeout=60)
        if r.ok and r.content:
            open(photo, "wb").write(r.content)
            fetched = url
            print("image", url, f"({len(r.content)} bytes)")
            break
        print(f"Could not fetch {url}: HTTP {r.status_code}")
    except requests.RequestException as exc:
        print(f"Could not fetch {url}: {exc}")
if not fetched:
    raise SystemExit("All image URLs failed.")

display(Image(filename=photo, width=360))
mime = "image/png" if photo.lower().endswith(".png") else "image/jpeg"
b64 = base64.b64encode(open(photo, "rb").read()).decode("ascii")
data_url = f"data:{mime};base64,{b64}"
VISION_PROMPT = "Describe this image in two sentences. Name the main objects you see."
print("=== prompt ===")
print(VISION_PROMPT)
print()
print("=== reply ===")
vision_run = chat(
    [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": VISION_PROMPT},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }
    ],
    max_tokens=512,
)
print(vision_run["text"].strip())
RUNS.append(
    {
        "name": "image-qa",
        "passed": None,
        "total": None,
        "completion_tokens": vision_run["completion_tokens"],
        "seconds": vision_run["seconds"],
        "tok_s": vision_run["tok_s"],
    }
)


## 9. Tokens/sec and peak VRAM

All demos, including the text chat. `nvidia-smi` is the VRAM counter (llama.cpp does not go through `torch.cuda.memory_allocated`).


In [ ]:
print(f"{'demo':<28} {'passed':>8} {'tokens':>8} {'sec':>8} {'tok/s':>8}")
print("-" * 64)
for row in RUNS:
    passed = "—" if row["passed"] is None else f"{row['passed']}/{row['total']}"
    print(
        f"{row['name']:<28} {passed:>8} {row['completion_tokens']:>8} "
        f"{row['seconds']:>8.1f} {row['tok_s']:>8.2f}"
    )
if RUNS:
    toks = sum(row["completion_tokens"] or 0 for row in RUNS)
    secs = sum(row["seconds"] or 0 for row in RUNS)
    print("-" * 64)
    print(f"{'ALL demos':<28} {'':>8} {toks:>8} {secs:>8.1f} {(toks / secs if secs else 0):>8.2f}")

print()
print("VRAM snapshots:")
for label, used in PEAK_VRAM:
    print(f"  {label}: {used}")
print()
print_vram("end")
print()
print("GPU status:")
subprocess.check_call(["nvidia-smi"])
print()
print("quant", QUANT, "| n-cpu-moe", N_CPU_MOE, "| llama.cpp", LLAMA_TAG, "| asset", ASSET_NAME)


## 10. ⏱ Benchmark

Placed **after** the demos and **before** the kill cell so the image question still uses the live server. Budget is about **five minutes** on a T4 with expert offload.

1. **Server timings** — three fixed prompts on `/v1/chat/completions` (short; medium ~1k-token prompt; long generation of 512 tokens). Prompt tok/s and generation tok/s come from llama.cpp's `timings` field.
2. **Stop `llama-server`** to free VRAM, then run the prebuilt **`llama-bench`** if the Unsloth tarball includes it (`-p 512 -n 128 -r 2`, same `-ngl` / `--n-cpu-moe` / `-fa`). Prints the markdown table (pp512 and tg128 tok/s). If `llama-bench` is missing, skip with a message — do not compile.


In [ ]:
def tokenize_len(text: str) -> int:
    try:
        r = requests.post(f"{SERVER_URL}/tokenize", json={"content": text}, timeout=30)
        if r.ok:
            data = r.json()
            toks = data.get("tokens") if isinstance(data, dict) else data
            if isinstance(toks, list):
                return len(toks)
    except requests.RequestException:
        pass
    return max(1, len(text) // 4)


def make_medium_prompt(target_tokens: int = 1000) -> str:
    unit = (
        "The quick brown fox jumps over the lazy dog. "
        "Pack my box with five dozen liquor jugs. "
    )
    text = unit * 8
    n = tokenize_len(text)
    reps = max(1, (target_tokens + n - 1) // n)
    text = unit * reps
    # Trim toward the target if we overshot a lot.
    while tokenize_len(text) > target_tokens + 80 and len(text) > len(unit):
        text = text[: int(len(text) * 0.9)]
    return (
        "Ignore the filler. Reply with one word: ok.\n\n" + text
    )


def bench_chat(name: str, prompt: str, max_tokens: int) -> dict:
    rec = chat([{"role": "user", "content": prompt}], max_tokens=max_tokens, timeout=240)
    timings = (rec.get("raw") or {}).get("timings") or {}
    prompt_n = timings.get("prompt_n") or rec.get("prompt_tokens") or 0
    prompt_tps = timings.get("prompt_per_second")
    gen_n = timings.get("predicted_n") or rec.get("completion_tokens") or 0
    gen_tps = timings.get("predicted_per_second") or rec.get("tok_s") or 0.0
    if not prompt_tps and prompt_n and timings.get("prompt_ms"):
        prompt_tps = 1000.0 * prompt_n / float(timings["prompt_ms"])
    row = {
        "name": name,
        "prompt_n": prompt_n,
        "prompt_tps": float(prompt_tps or 0.0),
        "gen_n": gen_n,
        "gen_tps": float(gen_tps or 0.0),
        "seconds": rec["seconds"],
    }
    print(
        f"{name}: prompt {row['prompt_n']} tok @ {row['prompt_tps']:.2f} tok/s | "
        f"gen {row['gen_n']} tok @ {row['gen_tps']:.2f} tok/s | {row['seconds']:.1f} s"
    )
    return row


print("=== server mini-benchmark (3 prompts) ===")
medium_prompt = make_medium_prompt(1000)
print(f"medium prompt ~{tokenize_len(medium_prompt)} tokens")
BENCH_ROWS = [
    bench_chat("short", "Reply with exactly the word ping.", 32),
    bench_chat("medium-pp", medium_prompt, 32),
    bench_chat(
        "long-gen",
        "Explain merge sort in clear prose with a tiny numeric example. Keep writing until the length cap.",
        512,
    ),
]
print()
print(f"{'name':<12} {'pp toks':>8} {'pp tok/s':>10} {'tg toks':>8} {'tg tok/s':>10} {'sec':>8}")
print("-" * 62)
for row in BENCH_ROWS:
    print(
        f"{row['name']:<12} {row['prompt_n']:>8} {row['prompt_tps']:>10.2f} "
        f"{row['gen_n']:>8} {row['gen_tps']:>10.2f} {row['seconds']:>8.1f}"
    )
print_vram("after server bench")


In [ ]:
def find_tool(root: str, name: str):
    direct = os.path.join(os.path.dirname(LLAMA_SERVER), name)
    if os.path.isfile(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if name in files:
            return os.path.join(dirpath, name)
    return None


def tool_help(path: str) -> str:
    try:
        return subprocess.check_output(
            [path, "-h"], env=ENV, text=True, stderr=subprocess.STDOUT, timeout=20
        )
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError) as exc:
        out = getattr(exc, "output", None) or str(exc)
        return out if isinstance(out, str) else str(out)


print("=== llama-bench (stop server first to free VRAM) ===")
stop_llama_server()
print_vram("after stop, before llama-bench")

llama_bench = find_tool(BIN_DIR, "llama-bench")
if not llama_bench:
    print(
        "llama-bench not found in the Unsloth CUDA 12 tarball. "
        "Skipping (do not compile llama.cpp). Server timings above still stand."
    )
else:
    os.chmod(llama_bench, os.stat(llama_bench).st_mode | 0o111)
    help_text = tool_help(llama_bench)
    bench_cmd = [
        llama_bench,
        "-m",
        MODEL_PATH,
        "-ngl",
        str(N_GPU_LAYERS),
        "-p",
        "512",
        "-n",
        "128",
        "-r",
        "2",
    ]
    if "n-cpu-moe" in help_text:
        bench_cmd.extend(["--n-cpu-moe", str(N_CPU_MOE)])
    elif "--cpu-moe" in help_text:
        print("llama-bench has no --n-cpu-moe; using --cpu-moe (all experts on CPU) so T4 VRAM fits.")
        bench_cmd.append("--cpu-moe")
    else:
        print(
            "llama-bench has no --n-cpu-moe; still running with -ngl",N_GPU_LAYERS,
            "(may OOM on T4).",
        )
    if "-fa" in help_text or "--flash-attn" in help_text:
        bench_cmd.extend(["-fa", "auto"] if "auto" in help_text else ["-fa", "1"])
    if "-o" in help_text or "--output" in help_text:
        bench_cmd.extend(["-o", "md"])
    print("running", " ".join(bench_cmd))
    try:
        proc = subprocess.run(
            bench_cmd,
            env=ENV,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=240,
        )
        print(proc.stdout[-8000:] if proc.stdout else "(no llama-bench output)")
        if proc.returncode != 0:
            print("llama-bench exit", proc.returncode)
    except subprocess.TimeoutExpired:
        print("llama-bench hit the 240 s cap (T4 expert-offload budget). Partial output is not used.")
    except OSError as exc:
        print("llama-bench could not start:", exc)

print_vram("after llama-bench")
print()
print("=== benchmark summary ===")
print(f"{'name':<12} {'pp toks':>8} {'pp tok/s':>10} {'tg toks':>8} {'tg tok/s':>10} {'sec':>8}")
print("-" * 62)
for row in BENCH_ROWS:
    print(
        f"{row['name']:<12} {row['prompt_n']:>8} {row['prompt_tps']:>10.2f} "
        f"{row['gen_n']:>8} {row['gen_tps']:>10.2f} {row['seconds']:>8.1f}"
    )
print()
print("Peak VRAM snapshots (nvidia-smi used):")
for label, used in PEAK_VRAM:
    print(f"  {label}: {used}")
print("quant", QUANT, "| n-cpu-moe", N_CPU_MOE, "| llama-bench", llama_bench or "missing")


## 11. Caveats and cleanup

**What this notebook is for.** Free-T4 inference of Unsloth's **UD-Q3_K_XL** Inkling 2 Mini GGUF (fallback **UD-IQ3_XXS**) through Unsloth's **prebuilt** CUDA 12 `llama-server`: a reasoning chat, three **executed** coding tests, an image question, and a short **benchmark** (`llama-bench` pp512/tg128 plus three server `timings` prompts).

**What it is not.** A source build of llama.cpp (do not compile on Colab T4). Mainline llama.cpp / `Mainline_Support/*.gguf_file` (needs unmerged [PR #32704](https://github.com/ggml-org/llama.cpp/pull/32704)). Full 1M context. Full GPU offload of the 13.63 GB file on 15 GB VRAM.

**T4 / expert offload.** Default `--n-cpu-moe 16` keeps the last 16 of 24 MoE layers' experts in RAM via mmap (no `--mlock`). That is conservative: raise toward 24 if you OOM; lower toward 8 if VRAM has headroom. Expert offload on Colab's ~2 CPU cores is **slow** compared with a full-GPU 3-bit run on a larger card. Tokens/sec above is a measurement from this session if you ran it, not a published card number.

**If Q3_K_XL cannot work.** Set `QUANT = "UD-IQ3_XXS"` (11.66 GB) in the download cell and rerun from there. The notebook prints that fallback in the download cell when it is selected.

**HF token.** Public repo. Cancel Colab `HF_TOKEN` prompts. `HF_HUB_DISABLE_IMPLICIT_TOKEN=1`.

**Not executed here.** This agent VM has no GPU. Notebook JSON and the Unsloth CUDA 12 `sm_75` asset URL / manifest were validated; a full T4 `llama-server` load was not run in this environment.

**Do not merge from this agent.** Leave the PR open for a Colab T4 PASS.

**Links:** [Unsloth GGUF](https://huggingface.co/unsloth/Inkling-2-Mini-GGUF) · [base card](https://huggingface.co/thinkingmachines/Inkling-2-Mini) · [run guide](https://unsloth.ai/docs/models/inkling-2-mini) · [prebuilts](https://github.com/unslothai/llama.cpp/releases) · [use cases](https://github.com/donvito/notebooks/blob/melvin/inkling-2-mini-ud-q3-gguf-cea2/docs/Inkling-2-Mini-UD-Q3-GGUF-llamacpp.md)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
stop_llama_server()
print_vram("after cleanup")
